# 🚀 Hybrid RAG with Microsoft phi-4-mini & SQLite FTS5 (Zero-Cloud SLM)

> **Author:** Çağrı Giray Keşan ([@Cagrik34](https://github.com/Cagrik34))  
> **Focus:** Small Language Models (SLMs), SQLite FTS5 BM25, Dense Embeddings, Reciprocal Rank Fusion (RRF)

---

## 📌 1. Motivation: The Keyword Recall Dilemma in Local SLMs
Standard RAG architectures relying purely on dense vector embeddings often fail to retrieve exact numerical tokens (e.g., `2,340,000 TL`, contract codes, account numbers).  
Conversely, sparse lexical search (BM25) misses semantic synonyms and paraphrased questions.

This cookbook demonstrates how to implement a **high-speed, in-memory Hybrid Retrieval engine** combining:
1. **Dense Vectors** (Cosine Similarity)
2. **Sparse Lexical Search** (SQLite FTS5 BM25)
3. **Reciprocal Rank Fusion (RRF, $k=60$)**
4. **Grounded Citation Generation (`[1]`, `[2]`)** with Microsoft `phi-4-mini`.

### References
- Cormack, Clarke & Buettcher. ["Reciprocal Rank Fusion outperforms Condorcet and Individual Rank Learning Methods."](https://dl.acm.org/doi/10.1145/1571941.1572114) SIGIR 2009.
- Robertson & Zaragoza. "The Probabilistic Relevance Framework: BM25 and Beyond." Foundations and Trends in IR, 2009.
- [SQLite FTS5 documentation](https://www.sqlite.org/fts5.html)
- [Microsoft Foundry Local](https://github.com/microsoft/Foundry-Local)
- [Phi-4-mini model card](https://huggingface.co/microsoft/Phi-4-mini-instruct)

> **Note on benchmarks:** this notebook does not ship a formal faithfulness/accuracy benchmark. Retrieval quality on your own corpus should be measured independently (e.g. Recall@k, citation-coverage) before quoting any accuracy figure elsewhere.


In [1]:
import hashlib
import logging
import os
import sqlite3
import numpy as np
from typing import List, Tuple, Dict, Any

RRF_K = 60
TOP_K = 2
EMBEDDING_DIM = 256
FOUNDRY_MODEL_ALIAS = "phi-4-mini"

logger = logging.getLogger(__name__)
print("✅ Core dependencies loaded successfully.")


✅ Core dependencies loaded successfully.


## 🧮 2. Real, Reproducible Embeddings (Local Hashing Fallback + Foundry Local)

To keep this notebook fully reproducible offline (no external model download required), dense
vectors are computed from the *actual chunk text* using a deterministic feature-hashing embedding
(a real, if simple, embedding technique -- not a hand-picked mock). When the
[`foundry-local-sdk`](https://pypi.org/project/foundry-local-sdk/) package **and** a running
Foundry Local instance with `phi-4-mini` are available, `embed()` transparently uses the real
model embeddings instead.

In [2]:
def hash_embedding(text: str, dim: int = EMBEDDING_DIM) -> List[float]:
    """Deterministic, dependency-free local embedding via feature hashing."""
    vec = np.zeros(dim, dtype=np.float32)
    for token in text.lower().split():
        digest = hashlib.sha256(token.encode("utf-8")).digest()
        bucket = int.from_bytes(digest[:4], "big") % dim
        sign = 1.0 if digest[4] % 2 == 0 else -1.0
        vec[bucket] += sign
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec.tolist()


class FoundryLocalUnavailable(Exception):
    """Raised when the Foundry Local SDK/runtime cannot service a request."""


def try_get_foundry_embedding(text: str, alias: str = FOUNDRY_MODEL_ALIAS) -> List[float]:
    try:
        from foundry_local_sdk import Configuration, FoundryLocalManager
    except ImportError as exc:
        raise FoundryLocalUnavailable(f"foundry-local-sdk is not installed: {exc}") from exc
    try:
        config = Configuration(app_name="phi-cookbook-hybrid-rag")
        if FoundryLocalManager.instance is None:
            FoundryLocalManager.initialize(config)
        manager = FoundryLocalManager.instance
        model = manager.catalog.get_model(alias)
        if model is None:
            raise FoundryLocalUnavailable(f"Model alias '{alias}' not found in local catalog.")
        model.load()
        response = model.get_embedding_client().generate_embedding(text)
        return list(response.data[0].embedding)
    except FoundryLocalUnavailable:
        raise
    except Exception as exc:
        raise FoundryLocalUnavailable(f"Foundry Local runtime unavailable: {exc}") from exc


def try_generate_with_foundry(prompt: str, alias: str = FOUNDRY_MODEL_ALIAS) -> str:
    try:
        from foundry_local_sdk import Configuration, FoundryLocalManager
    except ImportError as exc:
        raise FoundryLocalUnavailable(f"foundry-local-sdk is not installed: {exc}") from exc
    try:
        config = Configuration(app_name="phi-cookbook-hybrid-rag")
        if FoundryLocalManager.instance is None:
            FoundryLocalManager.initialize(config)
        manager = FoundryLocalManager.instance
        model = manager.catalog.get_model(alias)
        if model is None:
            raise FoundryLocalUnavailable(f"Model alias '{alias}' not found in local catalog.")
        model.load()
        completion = model.get_chat_client().complete_chat([{"role": "user", "content": prompt}])
        return completion.choices[0].message.content
    except FoundryLocalUnavailable:
        raise
    except Exception as exc:
        raise FoundryLocalUnavailable(f"Foundry Local runtime unavailable: {exc}") from exc


def embed(text: str) -> List[float]:
    """Real Foundry Local (phi-4-mini) embedding if available, else local hash fallback."""
    try:
        return try_get_foundry_embedding(text)
    except FoundryLocalUnavailable as exc:
        logger.info("Falling back to local hash embedding (%s)", exc)
        return hash_embedding(text)

print("✅ Embedding helpers ready (Foundry Local if available, else local hashing fallback).")


✅ Embedding helpers ready (Foundry Local if available, else local hashing fallback).


## 🏗️ 3. Dual SQLite Schema (Dense Vectors + Virtual FTS5 BM25 Table)

In [3]:
class LocalHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    chunk_index INTEGER NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                )
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    chunk_index UNINDEXED,
                    tokenize='unicode61'
                )
            """)

    def insert_chunk(self, source_file: str, chunk_index: int, content: str, embedding: List[float]) -> None:
        vec = np.array(embedding, dtype=np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm

        with self.conn:
            self.conn.execute(
                "INSERT INTO document_chunks (source_file, chunk_index, content, embedding) VALUES (?, ?, ?, ?)",
                (source_file, chunk_index, content, vec.tobytes())
            )
            self.conn.execute(
                "INSERT INTO document_chunks_fts (content, source_file, chunk_index) VALUES (?, ?, ?)",
                (content, source_file, str(chunk_index))
            )

    def search_dense(self, query_embedding: List[float], top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        q_vec = np.array(query_embedding, dtype=np.float32)
        q_norm = np.linalg.norm(q_vec)
        if q_norm > 0:
            q_vec = q_vec / q_norm

        cursor = self.conn.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            similarity = float(np.dot(q_vec, doc_vec))
            results.append((doc_id, src, content, similarity))
        results.sort(key=lambda x: x[3], reverse=True)
        return results[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        clean_tokens = [t for t in query_text.replace("'", "").replace('"', '').split() if len(t) > 1]
        if not clean_tokens:
            return []
        fts_query = " OR ".join(f'"{t}"' for t in clean_tokens)
        cursor = self.conn.execute(
            "SELECT rowid, source_file, content, rank FROM document_chunks_fts WHERE document_chunks_fts MATCH ? ORDER BY rank LIMIT ?",
            (fts_query, top_k)
        )
        results = []
        for doc_id, src, content, bm25_rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(bm25_rank)))
            results.append((doc_id, src, content, bm25_score))
        return results

    def hybrid_search(self, query_text: str, query_embedding: List[float], top_k: int = TOP_K) -> List[Dict[str, Any]]:
        """RRF fusion: RRF_score(d) = sum(1 / (k + rank_dense(d)), 1 / (k + rank_sparse(d)))"""
        dense_hits = self.search_dense(query_embedding, top_k=10)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=10)
        fused_scores = {}
        chunk_map = {}

        for rank, (doc_id, src, content, sim) in enumerate(dense_hits, start=1):
            key = f"{src}::{content[:50]}"
            chunk_map[key] = (src, content, "vector")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        for rank, (doc_id, src, content, bm25) in enumerate(sparse_hits, start=1):
            key = f"{src}::{content[:50]}"
            if key not in chunk_map:
                chunk_map[key] = (src, content, "bm25")
            else:
                chunk_map[key] = (src, content, "hybrid")
            fused_scores[key] = fused_scores.get(key, 0.0) + (1.0 / (RRF_K + rank))

        sorted_keys = sorted(fused_scores.keys(), key=lambda k: fused_scores[k], reverse=True)[:top_k]
        output = []
        for citation_idx, key in enumerate(sorted_keys, start=1):
            src, content, match_type = chunk_map[key]
            output.append({
                "citation_index": citation_idx,
                "source_file": src,
                "content": content,
                "rrf_score": fused_scores[key],
                "match_type": match_type
            })
        return output

print("✅ LocalHybridRAGStore class compiled successfully.")


✅ LocalHybridRAGStore class compiled successfully.


## 📊 4. Sample Ingestion & Hybrid Retrieval

In [4]:
store = LocalHybridRAGStore()

# Embeddings are computed from the actual chunk text via embed(), not hand-picked
# constants, so retrieval genuinely reflects the content and is reproducible from scratch.
sample_docs = [
    ("q3_financial_report.pdf", 0, "CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers."),
    ("architecture_specs.md", 0, "Zenith AI leverages Microsoft phi-4-mini (3.8B parameters) for local zero-cloud inference."),
    ("hr_policy_2026.docx", 0, "Remote work expense allowance is capped at 15,000 TL per employee quarterly.")
]

for src, idx, content in sample_docs:
    store.insert_chunk(src, idx, content, embed(content))

query = "What is the total allocated budget for the CodePulse project?"
query_vec = embed(query)

results = store.hybrid_search(query, query_vec, top_k=2)
for res in results:
    print(f"[{res['citation_index']}] {res['source_file']} ({res['match_type'].upper()}) -> Score: {res['rrf_score']:.4f}")
    print(f"    Content: {res['content']}\n")


[1] q3_financial_report.pdf (HYBRID) -> Score: 0.0328
    Content: CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers.

[2] hr_policy_2026.docx (HYBRID) -> Score: 0.0323
    Content: Remote work expense allowance is capped at 15,000 TL per employee quarterly.



## 📝 5. Grounded Prompt Formulation for Microsoft phi-4-mini

In [5]:
def construct_grounded_prompt(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for chunk in retrieved_chunks:
        context_blocks.append(f"[{chunk['citation_index']}] (Source: {chunk['source_file']})\n{chunk['content']}")
    context_str = "\n\n".join(context_blocks)

    return f"""You are Zenith AI, an enterprise-grade local assistant.
Answer the user query strictly based on the provided context below.
Every factual claim must cite its source index like [1] or [2].
If the context does not contain the answer, respond: 'This information is not present in the indexed documents.'

--- CONTEXT ---
{context_str}
--- END CONTEXT ---

User Query: {query}
Answer:"""

prompt = construct_grounded_prompt(query, results)
print(prompt)


You are Zenith AI, an enterprise-grade local assistant.
Answer the user query strictly based on the provided context below.
Every factual claim must cite its source index like [1] or [2].
If the context does not contain the answer, respond: 'This information is not present in the indexed documents.'

--- CONTEXT ---
[1] (Source: q3_financial_report.pdf)
CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers.

[2] (Source: hr_policy_2026.docx)
Remote work expense allowance is capped at 15,000 TL per employee quarterly.
--- END CONTEXT ---

User Query: What is the total allocated budget for the CodePulse project?
Answer:


## 🤖 6. Grounded Generation with phi-4-mini (Foundry Local)

This step performs a **real** call to `phi-4-mini` through Microsoft Foundry Local when the SDK
and a running local model are available. If not (e.g. this hosted notebook environment), it fails
gracefully and reports why, rather than fabricating a response.

In [6]:
try:
    answer = try_generate_with_foundry(prompt)
    print("✅ phi-4-mini response:\n")
    print(answer)
except FoundryLocalUnavailable as exc:
    reason = str(exc).splitlines()[0]
    print(f"⚠️  Foundry Local generation unavailable in this environment: {reason}")
    print("Install `foundry-local-sdk`, install the Foundry Local app, and run")
    print("`foundry model run phi-4-mini` to enable real generation.")


⚠️  Foundry Local generation unavailable in this environment: foundry-local-sdk is not installed: No module named 'foundry_local_sdk'
Install `foundry-local-sdk`, install the Foundry Local app, and run
`foundry model run phi-4-mini` to enable real generation.
